In [66]:
import pandas as pd
from typing import TypedDict, Dict, Any
from langgraph.graph import StateGraph,END
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
import os
from dotenv import load_dotenv
load_dotenv()

True

In [78]:
groq_api=os.getenv("GROQ_API_KEY")

In [79]:
claims_df=pd.read_csv("/Users/uttamtiwari/Desktop/agentic rag/data/warranty_claims.csv")
loader=PyPDFLoader("/Users/uttamtiwari/Desktop/agentic rag/data/AutoDrive_Warranty_Policy_2025.pdf")
policy_docs=loader.load()
policy_text=" ".join([doc.page_content for doc in policy_docs])

In [80]:
llm=ChatGroq(
    model="openai/gpt-oss-120b",

)


In [81]:
class ClaimState(TypedDict):
    claim :Dict[str,any]
    policy_check: str
    fraud_score: float
    evidence: str
    decision:str

In [82]:
def policy_check_agent(state:ClaimState) ->ClaimState:
    claim = state["claim"]
    vtype="Four-wheeler" if "Four-wheeler" in claim["model"] else "Two-wheeler"
    prompt=f""" you are an warrenty compailence officer warrenty check on basis of policy manual {policy_text}
    vechile type:{vtype}
    Claim details: {claim}
    Based on warranty days, mileage, and covered parts, is this claim covered under the policy?
    Answer with: "Covered by policy" or "Not covered by policy".
    """
    responce=llm.invoke(prompt)
   
    state['policy_check']=responce.content.strip()
    return state

In [83]:
def fraud_scoring_agent(state: ClaimState) -> ClaimState:
    claim=state["claim"]

    prompt=f"""

    Y ou are a fraud detection expert.

    Policy manual: (policy_text)

    Clain details:

    {claim}
    Policy validation: {state['policy_check']}
    Analyze whether this claim looks fraudulent.
    Return ONLY a number between 0 and 1 (fraud likelihood score)."""
    response=llm.invoke(prompt)
    try:
        score=float(response.content.strip())
    except:
        score = 0.5
    state ["fraud_score"]=score
    return state



In [84]:
def evidence_collector_agent(state: ClaimState) -> ClaimState:
    claim = state["claim"]
    prompt = f"""
    You are tasked with collecting evidence for claim review.
    Policy manual:
    {policy_text}
    
    Claim details:
    {claim}
    Fraud score: {state['fraud_score']}
    
    Compare claim against the policy manual and fraud indicators.
    List any red flags or violations found. If none, say "No issues".
    """
    response = llm.invoke(prompt)
    state["evidence"] = response.content.strip()
    return state


In [85]:
def action_agent(state: ClaimState) -> ClaimState:
 
    claim = state.get("claim", {})
    policy_check = state.get("policy_check", "")
    fraud_score = state.get("fraud_score", 0.0)
    evidence = state.get("evidence", "")

    prompt = f"""
    You are a warranty adjudicator. Given the following information about a warranty claim, choose one of three actions: "Approve claim", "Reject claim", or "Escalate to HITL" (human-in-the-loop for manual review).

    Provide your answer as a single decision on the first line, and then a short (1-2 sentence) justification on the following line.

    Policy manual (for reference):
    {policy_text}

    Claim details: {claim}

    Policy check result: {policy_check}
    Fraud score (0-1): {fraud_score}
    Evidence / red flags found: {evidence}

    Important: If the policy_check indicates the claim is "Not covered by policy" or the evidence highlights a direct policy violation (e.g., part not covered), prefer "Reject claim" unless strong justification exists to approve. If the evidence is ambiguous, but fraud score is moderately high (>0.5), choose "Escalate to HITL".
    """

    decision_text = ""
    try:
        response = llm.invoke(prompt)
        res_text = response.content.strip()
        
        first_line = res_text.splitlines()[0].strip()
        normalized = first_line.lower()
        if "approve" in normalized:
            decision_text = "Approve claim"
        elif "reject" in normalized:
            decision_text = "Reject claim"
        elif "escalate" in normalized or "hitl" in normalized or "human" in normalized:
            decision_text = "Escalate to HITL"
        else:
            
            decision_text = "" 
           
        state.setdefault("trace", []).append({
            "agent": "action_agent",
            "prompt": prompt,
            "response": res_text,
        })
    except Exception:
      
        res_text = ""

    
    if not decision_text:
        if policy_check == "Not covered by policy":
            decision_text = "Reject claim"
        elif fraud_score > 0.5:
            decision_text = "Escalate to HITL"
        else:
            decision_text = "Approve claim"

        state.setdefault("trace", []).append({
            "agent": "action_agent",
            "prompt": "(rule-based fallback)",
            "response": decision_text,
        })

    state["decision"] = decision_text
    return state

In [86]:
graph = StateGraph(ClaimState)

graph.add_node("PolicyCheck", policy_check_agent)
graph.add_node("FraudScoring", fraud_scoring_agent)
graph.add_node("EvidenceCollector", evidence_collector_agent)
graph.add_node("Action", action_agent)

graph.set_entry_point("PolicyCheck")
graph.add_edge("PolicyCheck", "FraudScoring")
graph.add_edge("FraudScoring", "EvidenceCollector")
graph.add_edge("EvidenceCollector", "Action")
graph.add_edge("Action", END)

app=graph.compile()

In [87]:
from IPython.display import display, Markdown

# Get the Mermaid code
mermaid_code = app.get_graph().draw_mermaid()

# Display in Jupyter
display(Markdown(f"```mermaid\n{mermaid_code}\n```"))

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	PolicyCheck(PolicyCheck)
	FraudScoring(FraudScoring)
	EvidenceCollector(EvidenceCollector)
	Action(Action)
	__end__([<p>__end__</p>]):::last
	EvidenceCollector --> Action;
	FraudScoring --> EvidenceCollector;
	PolicyCheck --> FraudScoring;
	__start__ --> PolicyCheck;
	Action --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [77]:
results = []

for idx, row in claims_df.iterrows():
    state = app.invoke({"claim": row.to_dict()})
    results.append({
        "claim_id": row["claim_id"],
        "model": row["model"],
        "decision": state["decision"],
        "policy_check": state["policy_check"],
        "fraud_score": state["fraud_score"],
        "evidence": state["evidence"]
    })

results_df = pd.DataFrame(results)

In [88]:
results_df

,claim_id,model,decision,policy_check,fraud_score,evidence
0,CLM-DEMO-1,"Volt-X (Four-Wheeler, EV Sedan)",Reject claim,Not covered by policy,0.86,**Red Flags / Policy Violations**\n\n| Issue |...


In [89]:
state


{'claim': {'claim_id': 'CLM-DEMO-1',
  'model': 'Volt-X (Four-Wheeler, EV Sedan)',
  'purchase_date': '8/8/2024',
  'claim_date': '5/24/2025',
  'days_since_purchase': 289,
  'mileage': 7363,
  'part_replaced': 'Headlight',
  'part_cost': 231,
  'labor_cost': 199,
  'total_cost': 430,
  'invoice_present': 1,
  'image_present': 1,
  'previous_claims': 1},
 'policy_check': 'Not covered by policy',
 'fraud_score': 0.86,
 'evidence': '**Red Flags / Policy Violations**\n\n| Issue | Policy Requirement | Claim Detail | Why it’s a problem |\n|-------|--------------------|--------------|--------------------|\n| **Part not covered** | Covered parts for Four‑Wheelers: Brake Pad, Airbag, Fuel Pump, Battery, Sensor, ECU, Alternator. *(Headlight is only listed for Two‑Wheelers.)* | Part replaced: **Headlight** | The claimed part is outside the list of approved components for a four‑wheel vehicle, so the claim is **invalid** under the warranty coverage rules. |\n| **Potential fraud indicator (high fr